In [12]:
from sklearn.datasets import fetch_openml
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score
from scipy.stats import mode


dataset = fetch_openml("mnist_784")
X = dataset['data']
y = dataset['target']


mask = np.array([True] * 60000 + [False] * 10000)
np.random.shuffle(mask)
X_train = X[mask]
X_test = X[~mask]

y_train = y[mask]
y_test = y[~mask]
'''
clf = DecisionTreeClassifier()
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
'''
           

'\nclf = DecisionTreeClassifier()\nclf.fit(X_train, y_train)\n\ny_pred = clf.predict(X_test)\n'

In [13]:
#print(classification_report(y_test, y_pred))
#print(accuracy_score(y_test,y_pred))

The following is my implementation of a pseudo random forest. This is wrong because each tree is trained on a specific subset of features (instead of having all features available and selecting a random subset at each split).

In [ ]:
from collections import Counter

class MyRandomForestClassifier():
    def __init__(self, n_estimators, max_features=None):
        self.n_estimators = n_estimators
        self.max_features = max_features
        self.clf_list = []

        pass

    def fit(self, X, y):

        N = len(X)
        num_f = len(X.iloc[0])

        if self.max_features is None:
            self.max_features = int(np.sqrt(num_f)) 

        self.feature_subsets = [] 

        for i in range(self.n_estimators):

            Test_indices = np.random.choice(N, size = N, replace=True)
            Feature_indices = np.random.choice(num_f, size=self.max_features, replace=False)

            X_test_subset = X.iloc[Test_indices, Feature_indices]
            y_test_subset = y.iloc[Test_indices]

            clf = DecisionTreeClassifier()
            clf.fit(X_test_subset, y_test_subset)

            self.clf_list.append(clf)
            self.feature_subsets.append(Feature_indices)

        return self.clf_list

    def predict(self, X):

        y_pred_list = []

        for t, f_index in zip(self.clf_list, self.feature_subsets):
            y_pred = t.predict(X_test.iloc[:, f_index])
            y_pred_list.append(y_pred)


        y_pred_list = np.array(y_pred_list).T
        preds = []

        for i in y_pred_list:
            preds.append(Counter(i).most_common(1)[0][0])
        #final_preds, _ = mode(y_pred_list, axis=1)
        #return final_preds.ravel()

        return np.array(preds).T

        
        

        


In [40]:
from collections import Counter

class MyRandomForestClassifier_sol():
    def __init__(self, n_estimators, max_features = 'sqrt'):
        self.trees = [DecisionTreeClassifier(max_features=max_features) 
                     for _ in range(n_estimators)]
        
    def fit(self, X, y):
        for tree in self.trees:
            subset = np.random.choice(range(X.shape[0]), size = X.shape[0], replace = True)

            tree.fit(X.iloc[subset], y.iloc[subset])

    def predict(self, X):
        predictions = [tree.predict(X) for tree in self.trees]
        print(predictions)

        
        preds = []
        for i in np.array(predictions).T:
            preds.append(Counter(i).most_common(1)[0][0])
        
        return preds

In [43]:
from sklearn.ensemble import RandomForestClassifier


rf = MyRandomForestClassifier(10, 28)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

rf_sol = MyRandomForestClassifier_sol(10, 28)
rf_sol.fit(X_train, y_train)
y_pred_rf_sol = rf_sol.predict(X_test)


rf_new = RandomForestClassifier(10)
rf_new.fit(X_train, y_train)
y_pred_rf_new = rf_new.predict(X_test)

print('built in impl: ',accuracy_score(y_pred_rf_new, y_test))
print('solution impl: ',accuracy_score(y_pred_rf_sol, y_test))
print('my impl: ',accuracy_score(y_pred_rf, y_test))


[array(['1', '5', '3', ..., '6', '0', '2'], shape=(10000,), dtype=object), array(['1', '5', '3', ..., '5', '0', '2'], shape=(10000,), dtype=object), array(['1', '5', '3', ..., '0', '0', '2'], shape=(10000,), dtype=object), array(['1', '5', '3', ..., '3', '0', '2'], shape=(10000,), dtype=object), array(['1', '8', '3', ..., '3', '0', '2'], shape=(10000,), dtype=object), array(['1', '5', '3', ..., '8', '0', '2'], shape=(10000,), dtype=object), array(['1', '1', '3', ..., '2', '0', '2'], shape=(10000,), dtype=object), array(['1', '5', '3', ..., '0', '0', '2'], shape=(10000,), dtype=object), array(['1', '6', '3', ..., '2', '0', '2'], shape=(10000,), dtype=object), array(['1', '2', '3', ..., '2', '0', '2'], shape=(10000,), dtype=object)]
built in impl:  0.9449
solution impl:  0.9463
my impl:  0.8684


In [44]:
for i in range(10, 101, 10):
    rf_sol = MyRandomForestClassifier_sol(i, 28)
    rf_sol.fit(X_train, y_train)
    y_pred_rf_sol = rf_sol.predict(X_test)
    print('solution impl: ',accuracy_score(y_pred_rf_sol, y_test))


[array(['1', '5', '3', ..., '5', '0', '2'], shape=(10000,), dtype=object), array(['1', '8', '3', ..., '2', '0', '2'], shape=(10000,), dtype=object), array(['1', '5', '3', ..., '5', '0', '2'], shape=(10000,), dtype=object), array(['1', '5', '3', ..., '5', '0', '2'], shape=(10000,), dtype=object), array(['1', '5', '3', ..., '0', '0', '2'], shape=(10000,), dtype=object), array(['1', '5', '3', ..., '2', '0', '2'], shape=(10000,), dtype=object), array(['1', '5', '3', ..., '6', '0', '2'], shape=(10000,), dtype=object), array(['1', '5', '3', ..., '6', '0', '2'], shape=(10000,), dtype=object), array(['1', '8', '3', ..., '3', '0', '2'], shape=(10000,), dtype=object), array(['1', '2', '3', ..., '5', '0', '2'], shape=(10000,), dtype=object)]
solution impl:  0.9475
[array(['1', '5', '3', ..., '6', '0', '2'], shape=(10000,), dtype=object), array(['1', '5', '3', ..., '6', '0', '2'], shape=(10000,), dtype=object), array(['1', '5', '3', ..., '0', '0', '2'], shape=(10000,), dtype=object), array(['1', '